## Set-Up der Umgebung

In [ ]:
# Install dependencies
!pip install openai wikipedia

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap !important;
    }
  </style>
  '''))

# Prevent horizontal scrolling in cell outputs
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
import os
import json
import openai
from google.colab import userdata

openai.api_key = userdata.get('OPENAI_API_KEY')

if not openai.api_key:
    raise ValueError("No OpenAI API key found! Make sure OPENAI_API_KEY is set as a Codespaces secret.")

print("✅ OpenAI API key loaded successfully!")

## Definiere Default-Tools

Im folgenden werden zwei einfache Tools definiert, die dem Agent von Anfang zur Verfügung stehen:

- **calculator:** Für das Lösen mathematischer Aufgaben, die als `expression` übergeben werden, z.B. `What is 12 * 12?`.
- **wiki_search:** Erlaubt dem Agenten, Zusammenfassungen aus Wikipedia-Artikeln zu erstellen, die sich auf die `query` beziehen.

Die erstellten Tools werden dem `tools`-Dictionary hinzugefügt, welches um neue Tools erweitert werden kann.

In [ ]:
# --- Tools ---

import wikipedia

def calculator(expression: str) -> str:
    """Evaluates basic math expressions."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

def wiki_search(query: str) -> str:
    """Fetches short summaries from Wikipedia."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Error: {e}"

tools = {
    "calculator": calculator,
    "wiki": wiki_search,
}

print("✅ Tools ready: calculator(), wiki_search()")

## Definiere die Anweisungen des Agenten

Die Definition der Anweisungen bzw. **System Message** des Agenten werden im folgenden als eigene Funktion ausgelagert, die in der Hauptfunktion des Agenten aufgerufen wird.

Das gibt uns mehr Flexiblität und hilft dabei, den Code übersichtlicher zu gestalten.

Die System Message setzt sich aus den folgenden Teilen zusammen:
- **tone**: Die Persönlichkeit des Agenten - wenn es keinen Input gibt, gibt es einen Default.
- **tools**: Anhand der *docstrings* (z.B. `"""Evaluates basic math expressions."""`) in den Tool-Funktionen wird automatisiert eine Liste von verfügbaren Tools zusammengestellt.
- **loop**: Der Agent wird angewiesen, stets den *Thought-Action-Observation-Loop* zu befolgen.
- **rules**: Weitere Anweisungen, z.B. *early stopping*, dass der Agent aus dem Loop aussteigt, wenn eine Antwort gefunden wurde.

In [ ]:
# --- System Message ---

def inject_system_message(agent_tone: str = None) -> str:
    # Assign the tone/personality if given, else fall back to default
    tone = agent_tone if agent_tone else "You are an intelligent agent that can reason step by step."

    # Dynamically gather tool descriptions to allow for new tools to be added later on
    tool_descriptions = "\n".join([f"- {name}(): {fn.__doc__ or 'no description'}" for name, fn in tools.items()])

    return f"""{tone}
    You have access to the following tools:
    {tool_descriptions}

    Always check first if you can solve the problem with one of the described tools!

    1) If you need to use a tool:
    Thought: describe your reasoning
    Action: <tool_name>("argument here")

    (The environment will then execute the tool and provide you with:
    Observation: <result>
    in the next turn.)

    2) If you already know the final answer and no more tools are needed:
    Thought: briefly explain that you know the answer
    Action: ANSWER(<final answer here>)

    Important rules:
    - You MUST treat each Observation as the authoritative result of the previous tool call.
    - After a tool call, you MUST use the latest Observation to inform your next Thought and Action.
    - ALWAYS start a new line for each 'Thought:', 'Action:' and 'Observation:'.
    - Use at most ONE tool invocation per response.
    - Do NOT repeat the same tool call multiple times with the same arguments.
    - Do NOT produce multiple ANSWER(...) actions in a single response.
    - Once you output `Action: ANSWER(...)`, your response must end immediately after that line.
    - You MUST NOT write any Observation lines yourself. The environment will show Observations after your Action.
    """

print("✅ Agent system message ready!")

## Definiere die Hauptfunktion des Agenten

Um den Agenten ausführen zu können, ist im folgenden die Funktion `run_agent` vorbereitet.

Diese nimmt die Anfrage (= *query*) des Users an den Agenten, die maximale Anzahl der *Thought-Action-Observation*-Schritte und einen optionalen *Tone* entgegen. Letzteres wirst du für eine spätere (Bonus-)Übung brauchen :).

Bearbeitet wird die Anfrage durch den Aufruf der OpenAI-Schnittstelle, welche die folgenden Argumente annimmt:
- **model**: Das zu verwendende LLM. In unserem Fall ist das `gpt-5-mini` völlig ausreichend.
- **instructions**: Die vorbereiteten Anweisungen für das Modell.
- **input**: Hier wird zum einen die *Query*, als auch der *Kontext* angefügt.

Die Antwort (= *response*) des Modells wird dann verarbeitet und je nach Inhalt entsprechend formattiert.

Der Kontext wird im weiteren Verlauf um die Ergebnisse bzw. *Observations* aus den vorangegangen Schritten erweitert.

In [ ]:
# --- Agent Function ---

from openai import OpenAI

def run_agent(query: str, max_steps: int = 3, agent_tone: str = None):
    print(f"User: {query}\n")

    client = OpenAI(
        api_key=openai.api_key
    )
    instructions = inject_system_message(agent_tone=agent_tone)
    context = ""

    for step in range(1, max_steps + 1):
        response = client.responses.create(
            model="gpt-5-mini",
            instructions=instructions,
            input=(
                f"User question: {query}\n"
                f"Previous observation(s):\n{context}"
            ),
        )

        text = response.output_text
        print(text)

        lines = text.splitlines()
        action_line = next((l for l in lines if l.startswith("Action:")), None)
        if not action_line:
            print("\nNo action detected. Stopping.")
            break

        action = action_line.replace("Action:", "").strip()

        if "ANSWER" in action:
            final = action.split("ANSWER")[-1].strip(" :()")
            print(f"\nFinal Answer: {final}")
            return

        if "(" in action and action.endswith(")"):
            tool_name = action.split("(")[0]
            tool_input = _unquote_string(action[len(tool_name) + 1:-1])

            if tool_name in tools:
                observation = tools[tool_name](tool_input) if tool_input else tools[tool_name]()
            else:
                observation = f"Unknown tool: {tool_name}"

            # print observation
            print(f"Observation: {observation}\n")
            context += f"\n{text}\nObservation: {observation}\n"
        else:
            print("\nCould not parse tool call. Stopping.")
            break

def _unquote_string(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ('"', "'"):
        return s[1:-1]
    return s

print("✅ Agent main function set up!")

## Probier's aus!

Führe die **Agenten-Funktion** mit verschiedenen Anfragen aus und teste die Verwendung der *Default-Tools* und des *ReAct-Loop*-Formats.

In [ ]:
run_agent("What is 1234 * 5678?")

In [ ]:
run_agent("When was the Eiffel Tower built?")

In [ ]:
# Your code here ...

# 🧩 Übung: Füge dein eigenes Tools hinzu

Jetzt bist du dran!

Füge ein **neues Tool** hinzu, das der Agent verwenden kann.  
Einige Ideen:
- Ein `reverse(text)`-Tool, das Wörter umkehrt.
- Ein `coin_flip()`-Tool, das zufällig zwischen „Kopf“ und „Zahl“ wählt.
- Ein Tool `word_count(text)`, das Wörter zählt.

---

### 🧠 Vorgehensweise

1. Definiere dein Tool und beschreibe dessen Funktion mit einem *docstring* (siehe Beispiel in nächster Code-Zelle).
2. Füge dein neues Tool dem `tools`-Dictionary hinzu.
3. Versuche, durch deine Prompts, den Agenten aufzufordern, das neue Tool zu verwenden!

## Beispiel: Word Count Tool

In [ ]:
def reverse(text: str) -> str:
    """Reverses the order of words in the input text."""
    try:
        return " ".join(reversed(text.split()))
    except Exception as e:
        return f"Error: {e}"

# Don't forget to add your new tool to the dictionary
tools["reverse"] = reverse

print("✅ Added new tool: reverse()")

In [ ]:
run_agent("Reverse the following text: 'Artificial intelligence enables creative problem solving!'")

In [ ]:
# Your code here ...

# 🔍 Übung: Multi-Tool Agent

Kannst du den Agenten dazu bringen, **mehrere Tools** zu kombinieren?

Zum Beispiel:
> "What is the population of Berlin divided by the population of Munich?"

Damit könnte der Agent...
1. das `wiki`-Tool zweimal verwenden (einmal pro Land), um die ungefähre Einwohnerzahl zu bestimmen und
2. mithilfe des `calculator`-Tools das Ergebnis ausrechnen.

**Tipp:** Du kannst mehr Schritte im ReAct-Loop zulassen, indem du das Argument `max_steps` in `run_agent()` erhöhst.

In [ ]:
run_agent("What is the population of Berlin divided by the population of Munich?", max_steps=5)

In [ ]:
# Your code here ...

# ⛅ Exkurs: Anbindung einer Open Source Wetter-API

Bisher haben wir Tools kennengelernt, die häufig eine einfache interne Logik ausführen (wie der `calculator` oder `reverse`). Das `wiki`-Tool wiederum nutzt das lokal installierte Python-Paket, um auf die **externe Wikipedia-API** zuzugreifen und dort Daten abzurufen. Hier kommuniziert der Agent sozusagen "nach außen".

Die Fähigkeit von Agenten, mit externen Diensten und APIs (*= Application Programming Interfaces*) zu interagieren, ist besonders spannend und birgt viel Potential. So können sie Informationen aus der realen Welt abrufen oder Aktionen ausführen, die über die unmittelbare Umgebung des Notebooks/lokalen Systems hinausgehen.

**Im Folgenden werden wir ein weiteres Tool erstellen, das Daten über eine externe API abruft: die Open-Meteo API.**

---

### Was ist Open-Meteo?

[Open-Meteo](https://open-meteo.com/) ist eine open source Wetter-API, die kostenlose und unbegrenzte Zugriffe auf Wetterdaten ermöglicht. Sie bietet globale Wettervorhersagen und historische Daten, basierend auf hochwertigen Wettermodellen.

### Aufbau des `weather`-Tools

Das `weather`-Tool, das wir implementieren, verwendet ein zweistufiges Verfahren, um aktuelle Wetterinformationen für eine angegebene Stadt abzurufen:

1.  **Geocoding:** Die Open-Meteo API benötigt Breitengrad (`latitude`) und Längengrad (`longitude`) einer Stadt, um Wetterdaten abzurufen. Da wir dem Tool nur den Namen einer Stadt (z.B. "Stuttgart") übergeben, benötigen wir einen Weg, diesen Namen in Koordinaten umzuwandeln. Hier kommt die Hilfsfunktion `_geocode_city` ins Spiel, welche ihrerseits die **Geocoding API** von Open-Meteo anspricht.
2.  **Wetterdaten abrufen:** Mit den erhaltenen Koordinaten wird dann die eigentliche **Wetter-API** von Open-Meteo aufgerufen, um die aktuellen Wetterbedingungen wie Temperatur, Windgeschwindigkeit und einen Wettercode zu erhalten.

In [ ]:
# Test: Was macht der Agent vor Hinzufügen des Tools?
run_agent("How's the weather in Stuttgart?")

In [ ]:
import requests

def _geocode_city(city: str) -> dict | None:
    """Fetches geocoding data for a given city from Open-Meteo's Geocoding API."""
    geocoding_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en&format=json"
    geocoding_response = requests.get(geocoding_url)
    geocoding_data = geocoding_response.json()

    if not geocoding_data or not geocoding_data.get('results'):
        print(f"Error: City '{city}' not found via Open-Meteo Geocoding.")
        return None

    return geocoding_data

In [ ]:
def weather(city: str) -> str:
    """Fetches current weather information for a given city using Open-Meteo APIs."""

    # Step 1: Use helper function to get geocoded data of city
    geocoded_city = _geocode_city(city=city)

    if geocoded_city:
        first_result = geocoded_city['results'][0]
        latitude = first_result['latitude']
        longitude = first_result['longitude']
        name = first_result['name']
        country = first_result['country']

        # Step 2: Use selected data from geocoding as input to fetch current weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true&forecast_days=1&timezone=auto"
        weather_response = requests.get(weather_url)
        weather_data = weather_response.json()

        if not weather_data or not weather_data.get('current_weather'):
            return f"Error: Could not retrieve current weather data for {name}, {country}."

        current_weather = weather_data['current_weather']
        temperature = current_weather['temperature']
        windspeed = current_weather['windspeed']
        weather_code = current_weather['weathercode']

        return (
            f"Current weather in {name}, {country} (Lat: {latitude}, Lon: {longitude}):\n"
            f"\tTemperature: {temperature}°C,\n"
            f"\tWind speed: {windspeed} km/h,\n"
            f"\tWeather code: {weather_code}"
        )

    return f"Could not fetch weather data for city {city}."

tools["weather"] = weather
print("✅ Added new tool: weather()")

In [ ]:
# Fragen wir nochmal (jetzt mit Unterstützung des Tools)
run_agent("How's the weather in Stuttgart?")

## Zeit für weitere Experimente:

- Was passiert, wenn du fiktionale Städte angibst?
- Welche Daten zum Wetter könnten noch interessant sein? Schau dich in der [Doku](https://open-meteo.com/en/docs) um und füge diese dem `weather`-Tool hinzu.

In [ ]:
# Your code here ...

# 🎭 Bonus: Verändere die Persönlichkeit des Agenten

Jetzt kommt das optionale `tone`-Argument der `run_agent()`-Funktion zum Einsatz! Verändere die Persönlichkeit des Agenten.  
Hier ein paar Inspirationen:

- ein **Detektiv** dem kein Rätsel zu schwer ist,  
- ein **mittelalterlicher Ritter**, oder
- ein **Digital Native** aus der Generation Z.

---

### 🧠 Vorgehensweise:

1. Erstelle eine Variable für die neue Persönlichkeit und weise diese beim Aufruf von `run_agent` dem Wert `agent_tone=<dein_neuer_tone>` zu:
    > teacher_tone = "You are a Sherlock-esque detective who explains your reasoning out loud before answering."
2. Probiere verschiedene *tones* aus und beobachte, wie sich der Agent verhält!

## Beispiel: Gen-Z Agent

In [ ]:
# Define a different personality tone than what is currently the default

gen_z_tone = "You are a typical Generation-Z young adult who explains your reasoning out loud (in internet/meme language) before answering."

In [ ]:
# Run the agent with example questions as querys and the Gen-Z tone setting

print("___ Gen-Z Agent ___")
run_agent(query="What is 1234 * 5678?", agent_tone=gen_z_tone)

print("------")
run_agent(query="When was the Eiffel Tower built?", agent_tone=gen_z_tone)

In [ ]:
# Your code here ...

# 🏁 Wrap-Up

🎉 Glückwunsch — du hast gerade einen funktionierenden KI-Agenten erstellt!

Du hast gelernt:
- Wie ein Agent mithilfe des **Thought–Action–Observation**-Loop folgert und handelt.  
- Wie **Tools**, zum Beispiel der Taschenrechner oder die Wikipedia-Suche, integriert werden.  
- Wie du das System mit deinen eigenen kreativen Ideen **erweitern** kannst.
- Wie man **externe APIs** anbindet, um den Agenten mit Echtzeitdaten zu versorgen.

---

💬 **Ausblick MCP (= Model Context Protocol)**

Die heutige Session hat dir gezeigt, wie ein einzelner Agent mit individuell definierten Tools arbeitet. MCP geht einen Schritt weiter und definiert ein standardisiertes Protokoll, über das Modelle konsistent auf Tools, Datenquellen und Systeme zugreifen können.

... dazu mehr im nächsten Community Afternoon 👀.

# ❤️ Vielen Dank für deine Teilnahme!